<a href="https://colab.research.google.com/github/mdmasoomcse/TNS-prcatice-sessions/blob/main/Feature%20Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering for Different Data Types

## 1. Setup and Data Generation

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures
from sklearn.feature_extraction.text import TfidfVectorizer

# Create a synthetic dataset with various data types
data = {
    'numerical_feature_1': np.random.rand(10) * 100, # Continuous numerical
    'numerical_feature_2': np.random.randint(0, 50, 10), # Discrete numerical
    'categorical_feature_1': ['A', 'B', 'A', 'C', 'B', 'A', 'C', 'B', 'A', 'C'], # Low cardinality categorical
    'categorical_feature_2': ['Red', 'Green', 'Blue', 'Red', 'Yellow', 'Green', 'Red', 'Blue', 'Yellow', 'Green'], # Medium cardinality categorical
    'text_feature': [
        'This is a sample sentence.',
        'Another sentence for text processing.',
        'Text features are important.',
        'Machine learning is fun.',
        'Sample text for analysis.',
        'More text for our model.',
        'Natural language processing.',
        'Data science rocks.',
        'Feature engineering is key.',
        'Final example sentence.'
    ],
    'datetime_feature': pd.to_datetime([
        '2023-01-01 10:30:00',
        '2023-01-01 11:00:00',
        '2023-01-02 12:15:00',
        '2023-01-02 13:45:00',
        '2023-01-03 14:00:00',
        '2023-01-03 15:30:00',
        '2023-01-04 16:00:00',
        '2023-01-04 17:15:00',
        '2023-01-05 18:30:00',
        '2023-01-05 19:45:00'
    ]),
    'target': np.random.randint(0, 2, 10) # Target variable
}

df = pd.DataFrame(data)
display(df.head())

,numerical_feature_1,numerical_feature_2,categorical_feature_1,categorical_feature_2,text_feature,datetime_feature,target
0,74.568275,12,A,Red,This is a sample sentence.,2023-01-01 10:30:00,0
1,3.283492,7,B,Green,Another sentence for text processing.,2023-01-01 11:00:00,0
2,78.923935,16,A,Blue,Text features are important.,2023-01-02 12:15:00,0
3,38.677302,31,C,Red,Machine learning is fun.,2023-01-02 13:45:00,1
4,62.945054,38,B,Yellow,Sample text for analysis.,2023-01-03 14:00:00,0


## 2. Numerical Feature Engineering

Numerical features can be transformed in various ways to improve model performance, such as scaling, binning, or creating polynomial features.

### 2.1 Scaling Numerical Features

Scaling helps to standardize the range of independent variables or features of the data. This is crucial for many machine learning algorithms (e.g., SVM, K-Means, Neural Networks) that are sensitive to the magnitude of the input features.

#### Min-Max Scaling (Normalization)

In [2]:
scaler_minmax = MinMaxScaler()
df['numerical_feature_1_minmax'] = scaler_minmax.fit_transform(df[['numerical_feature_1']])
display(df[['numerical_feature_1', 'numerical_feature_1_minmax']].head())

,numerical_feature_1,numerical_feature_1_minmax
0,74.568275,0.783655
1,3.283492,0.000000
2,78.923935,0.831538
3,38.677302,0.389095
4,62.945054,0.655877


#### Standard Scaling (Standardization)

In [3]:
scaler_standard = StandardScaler()
df['numerical_feature_1_standard'] = scaler_standard.fit_transform(df[['numerical_feature_1']])
display(df[['numerical_feature_1', 'numerical_feature_1_standard']].head())

,numerical_feature_1,numerical_feature_1_standard
0,74.568275,0.706581
1,3.283492,-2.108522
2,78.923935,0.878591
3,38.677302,-0.710787
4,62.945054,0.247569


### 2.2 Binning/Discretization

Binning converts continuous numerical variables into discrete categorical bins. This can help to capture non-linear relationships and reduce the impact of outliers.

In [4]:
df['numerical_feature_1_binned'] = pd.cut(df['numerical_feature_1'], bins=3, labels=['Low', 'Medium', 'High'])
display(df[['numerical_feature_1', 'numerical_feature_1_binned']].head())

,numerical_feature_1,numerical_feature_1_binned
0,74.568275,High
1,3.283492,Low
2,78.923935,High
3,38.677302,Medium
4,62.945054,Medium


### 2.3 Polynomial Features

Creating polynomial features involves generating new features by raising existing features to a power or by multiplying features together. This can help models capture non-linear relationships.

In [5]:
poly = PolynomialFeatures(degree=2, include_bias=False)
poly_features = poly.fit_transform(df[['numerical_feature_1', 'numerical_feature_2']])
poly_feature_names = poly.get_feature_names_out(['numerical_feature_1', 'numerical_feature_2'])
df_poly = pd.DataFrame(poly_features, columns=poly_feature_names)
df = pd.concat([df, df_poly], axis=1)
display(df[poly_feature_names].head())

,numerical_feature_1,numerical_feature_1,numerical_feature_2,numerical_feature_2,numerical_feature_1^2,numerical_feature_1 numerical_feature_2,numerical_feature_2^2
0,74.568275,74.568275,12,12.0,5560.427673,894.819303,144.0
1,3.283492,3.283492,7,7.0,10.781322,22.984447,49.0
2,78.923935,78.923935,16,16.0,6228.987475,1262.782956,256.0
3,38.677302,38.677302,31,31.0,1495.933688,1198.996361,961.0
4,62.945054,62.945054,38,38.0,3962.079851,2391.912060,1444.0


## 3. Categorical Feature Engineering

Machine learning models typically require numerical input, so categorical features need to be converted. Common techniques include One-Hot Encoding and Label Encoding.

### 3.1 One-Hot Encoding

One-Hot Encoding creates new binary features for each category in a categorical variable. This is suitable for nominal (unordered) categorical data.

In [6]:
encoder_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe_features = encoder_ohe.fit_transform(df[['categorical_feature_1']])
ohe_feature_names = encoder_ohe.get_feature_names_out(['categorical_feature_1'])
df_ohe = pd.DataFrame(ohe_features, columns=ohe_feature_names)
df = pd.concat([df, df_ohe], axis=1)
display(df[ohe_feature_names].head())

,categorical_feature_1_A,categorical_feature_1_B,categorical_feature_1_C
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,1.0,0.0,0.0
3,0.0,0.0,1.0
4,0.0,1.0,0.0


### 3.2 Label Encoding

Label Encoding assigns a unique integer to each category. This is suitable for ordinal (ordered) categorical data, but can imply an ordering that doesn't exist if used on nominal data.

In [7]:
encoder_le = LabelEncoder()
df['categorical_feature_2_encoded'] = encoder_le.fit_transform(df['categorical_feature_2'])
display(df[['categorical_feature_2', 'categorical_feature_2_encoded']].head())

,categorical_feature_2,categorical_feature_2_encoded
0,Red,2
1,Green,1
2,Blue,0
3,Red,2
4,Yellow,3


### 3.3 Frequency Encoding

Frequency encoding replaces each category with its frequency of occurrence in the dataset. This can be useful for high-cardinality categorical features.

In [8]:
frequency_map = df['categorical_feature_2'].value_counts(normalize=True).to_dict()
df['categorical_feature_2_freq_encoded'] = df['categorical_feature_2'].map(frequency_map)
display(df[['categorical_feature_2', 'categorical_feature_2_freq_encoded']].head())

,categorical_feature_2,categorical_feature_2_freq_encoded
0,Red,0.3
1,Green,0.3
2,Blue,0.2
3,Red,0.3
4,Yellow,0.2


## 4. Text Feature Engineering

Text data requires specialized techniques to convert into numerical representations that machine learning models can understand.

### 4.1 TF-IDF (Term Frequency-Inverse Document Frequency)

TF-IDF assigns a weight to each word in a document based on its frequency within the document and its rarity across all documents.

In [9]:
tfidf_vectorizer = TfidfVectorizer(max_features=5) # Limiting features for display
tfidf_features = tfidf_vectorizer.fit_transform(df['text_feature']).toarray()
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
df_tfidf = pd.DataFrame(tfidf_features, columns=[f'tfidf_{name}' for name in tfidf_feature_names])
df = pd.concat([df, df_tfidf], axis=1)
display(df[[col for col in df.columns if 'tfidf_' in col]].head())

,tfidf_for,tfidf_is,tfidf_processing,tfidf_sentence,tfidf_text
0,0.000000,0.707107,0.000000,0.707107,0.000000
1,0.494050,0.000000,0.564705,0.494050,0.439246
2,0.000000,0.000000,0.000000,0.000000,1.000000
3,0.000000,1.000000,0.000000,0.000000,0.000000
4,0.747341,0.000000,0.000000,0.000000,0.664440


### 4.2 Basic Text Statistics

Simple features like word count, character count, and average word length can also be informative.

In [10]:
df['word_count'] = df['text_feature'].apply(lambda x: len(str(x).split()))
df['char_count'] = df['text_feature'].apply(lambda x: len(str(x)))
display(df[['text_feature', 'word_count', 'char_count']].head())

,text_feature,word_count,char_count
0,This is a sample sentence.,5,26
1,Another sentence for text processing.,5,37
2,Text features are important.,4,28
3,Machine learning is fun.,4,24
4,Sample text for analysis.,4,25


## 5. Date/Time Feature Engineering

Date and time features can often contain rich information that can be extracted to improve model performance. This includes components like year, month, day, day of week, hour, etc.

In [11]:
df['year'] = df['datetime_feature'].dt.year
df['month'] = df['datetime_feature'].dt.month
df['day'] = df['datetime_feature'].dt.day
df['day_of_week'] = df['datetime_feature'].dt.dayofweek # Monday=0, Sunday=6
df['hour'] = df['datetime_feature'].dt.hour
df['is_weekend'] = (df['datetime_feature'].dt.dayofweek >= 5).astype(int) # Saturday=5, Sunday=6
display(df[['datetime_feature', 'year', 'month', 'day', 'day_of_week', 'hour', 'is_weekend']].head())

,datetime_feature,year,month,day,day_of_week,hour,is_weekend
0,2023-01-01 10:30:00,2023,1,1,6,10,1
1,2023-01-01 11:00:00,2023,1,1,6,11,1
2,2023-01-02 12:15:00,2023,1,2,0,12,0
3,2023-01-02 13:45:00,2023,1,2,0,13,0
4,2023-01-03 14:00:00,2023,1,3,1,14,0


## 6. Final Data with Engineered Features

Here's a look at the DataFrame with all the new engineered features. Note that some original features (like raw text or datetime) might be dropped before training a model, depending on the specific task.

In [12]:
display(df.head())

,numerical_feature_1,numerical_feature_2,categorical_feature_1,categorical_feature_2,text_feature,datetime_feature,target,numerical_feature_1_minmax,numerical_feature_1_standard,numerical_feature_1_binned,...,tfidf_sentence,tfidf_text,word_count,char_count,year,month,day,day_of_week,hour,is_weekend
0,74.568275,12,A,Red,This is a sample sentence.,2023-01-01 10:30:00,0,0.783655,0.706581,High,...,0.707107,0.000000,5,26,2023,1,1,6,10,1
1,3.283492,7,B,Green,Another sentence for text processing.,2023-01-01 11:00:00,0,0.000000,-2.108522,Low,...,0.494050,0.439246,5,37,2023,1,1,6,11,1
2,78.923935,16,A,Blue,Text features are important.,2023-01-02 12:15:00,0,0.831538,0.878591,High,...,0.000000,1.000000,4,28,2023,1,2,0,12,0
3,38.677302,31,C,Red,Machine learning is fun.,2023-01-02 13:45:00,1,0.389095,-0.710787,Medium,...,0.000000,0.000000,4,24,2023,1,2,0,13,0
4,62.945054,38,B,Yellow,Sample text for analysis.,2023-01-03 14:00:00,0,0.655877,0.247569,Medium,...,0.000000,0.664440,4,25,2023,1,3,1,14,0


In [13]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 33 columns):
 #   Column                                   Non-Null Count  Dtype         
---  ------                                   --------------  -----         
 0   numerical_feature_1                      10 non-null     float64       
 1   numerical_feature_2                      10 non-null     int64         
 2   categorical_feature_1                    10 non-null     object        
 3   categorical_feature_2                    10 non-null     object        
 4   text_feature                             10 non-null     object        
 5   datetime_feature                         10 non-null     datetime64[ns]
 6   target                                   10 non-null     int64         
 7   numerical_feature_1_minmax               10 non-null     float64       
 8   numerical_feature_1_standard             10 non-null     float64       
 9   numerical_feature_1_binned               10 no